# Weather preprocessing and merge





**Imports & Setup**  
We load core libraries (`pandas`, `numpy`, etc.) and utilities used throughout the notebook. These imports power I/O (CSV/Parquet), time handling, joins, interpolation, and plotting.

> **Why:** clean, preprocessing makes the downstream LSTM feature ingestion deterministic and auditable.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np





In [2]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("src")

In [3]:
df_switch_clean = pd.read_csv("data/powbal clean/powbal_clean.csv", low_memory=False)

In [4]:
from powbal.full_pipeline import apply_all_transformations
df_switch_transformed = apply_all_transformations(df_switch_clean)

In [5]:
from powbal.sequence_preparation import (
    extract_event_timing_features,
    build_sequences_for_modeling
)

In [6]:
df_switch_transformed = extract_event_timing_features(df_switch_transformed)

In [5]:
df_switch_transformed = pd.read_pickle("data/df_switch_transformed_1.pkl")

In [10]:
df_switch_clean = pd.read_csv("data/powbal clean/powbal_clean.csv", low_memory=False)

**Load customer base tables (Delhi / Mumbai).**  
Read city-specific customer files.

In [9]:
delhicust = pd.read_csv("data/powbal clean/delhi-customers.csv", low_memory=False)

In [10]:
mumbaicust = pd.read_csv("data/powbal clean/mumbai-customers.csv", low_memory=False)

**Load city–year weather slices.**  
We ingest hourly weather files (e.g. `weather-delhi-2023`, `weather-delhi-2024`, `weather-mumbai-2023`, `weather-mumbai-2024`). Columns typically include `timestamp`, wind (`wind_speed`, `wind_direction`), `temperature`, precipitation, short‑wave radiation (`surface_net_solar_radiation`/`_downwards`), `location`, and a `city` label.


In [11]:
weather23delhi = pd.read_csv("data/powbal clean/weather-delhi-2023.csv", low_memory=False)

In [12]:
weather24delhi = pd.read_csv("data/powbal clean/weather-delhi-2024.csv", low_memory=False)

In [13]:
weather23mumbai = pd.read_csv("data/powbal clean/weather-mumbai-2023.csv", low_memory=False)

In [14]:
weather24mumbai = pd.read_csv("data/powbal clean/weather-mumbai-2024.csv", low_memory=False)

In [11]:
weather24mumbai.head()

,timestamp,wind_speed,wind_direction,temperature,city,precipitation,location,surface_net_solar_radiation,surface_solar_radiation_downwards
0,2024-01-01 01:00:00.000000,3.536598,235.034510,20.425812,mumbai,0.0,POINT (72.97 18.899999999999995),0.0,0.0
1,2024-01-01 01:00:00.000000,4.879442,226.405153,21.919830,mumbai,0.0,POINT (72.86999999999998 18.799999999999994),0.0,0.0
2,2024-01-01 01:00:00.000000,4.246554,235.989930,21.010651,mumbai,0.0,POINT (72.97 18.799999999999994),0.0,0.0
3,2024-01-01 01:00:00.000000,4.157160,243.949795,20.296783,mumbai,0.0,POINT (72.97 18.699999999999992),0.0,0.0
4,2024-01-01 01:00:00.000000,4.067148,242.516064,20.201080,mumbai,0.0,POINT (72.97 18.59999999999999),0.0,0.0


In [ ]:
customer join

**Union of customer slices.**  
We stack the Delhi and Mumbai customer frames keeping the original schema (`CA_ID`, `Lat`, `Lon`, `city`).

**Range/coverage diagnostics.**  
Compute per‑city minima, maxima, and counts to confirm temporal coverage and detect outliers, keeping the original values intact.


In [15]:
# Aggiungo la città (se non già presente) e tolgo l'eventuale colonna-indice
if 'city' not in delhicust.columns:
    delhicust['city'] = 'delhi'
if 'Unnamed: 0' in delhicust.columns:
    delhicust = delhicust.drop(columns=['Unnamed: 0'])

if 'city' not in mumbaicust.columns:
    mumbaicust['city'] = 'mumbai'
if 'Unnamed: 0' in mumbaicust.columns:
    mumbaicust = mumbaicust.drop(columns=['Unnamed: 0'])

# Unione finale
customer_data = pd.concat([delhicust, mumbaicust], ignore_index=True)

# Tipi coerenti (mantengo i NOME COLONNE originali)
customer_data['CA_ID'] = customer_data['CA_ID'].astype(str).str.strip()
customer_data['Lat'] = pd.to_numeric(customer_data['Lat'], errors='coerce')
customer_data['Lon'] = pd.to_numeric(customer_data['Lon'], errors='coerce')

print(f"[customer_data] shape: {customer_data.shape[0]:,} × {customer_data.shape[1]}")
print("Colonne:", list(customer_data.columns))

# NaN %
nan_pct_customer = (customer_data.isna().mean() * 100).round(2).sort_values(ascending=False)
print("\nNaN % (prime 15):")
print(nan_pct_customer.head(15))

# Duplicati e cardinalità
print("\nDuplicati su CA_ID:", int(customer_data.duplicated(subset=['CA_ID']).sum()))
print("CA_ID unici:", customer_data['CA_ID'].nunique())

# Range geografico per città
print("\nRange Lat/Lon per città:")
print(customer_data.groupby('city')[['Lat','Lon']].agg(['min','max']))

# OUTLIER geografici (fuori India o coordinate nulle)
mask_outside_india = (~customer_data['Lat'].between(6, 38)) | (~customer_data['Lon'].between(68, 98))
mask_zero_zero = (customer_data['Lat'].fillna(0) == 0) & (customer_data['Lon'].fillna(0) == 0)
customer_outliers = customer_data.loc[mask_outside_india | mask_zero_zero, ['CA_ID','city','Lat','Lon']]

print(f"\nPossibili outlier customer: {len(customer_outliers)} righe (mostro le prime 10)")
customer_outliers.head(10)


[customer_data] shape: 514 × 4
Colonne: ['CA_ID', 'Lat', 'Lon', 'city']

NaN % (prime 15):
CA_ID    0.0
Lat      0.0
Lon      0.0
city     0.0
dtype: float64

Duplicati su CA_ID: 0
CA_ID unici: 514

Range Lat/Lon per città:
              Lat                   Lon            
              min        max        min         max
city                                               
delhi   28.623714  57.367808  76.974229  154.320738
mumbai   0.000000  19.286050   0.000000   72.911674

Possibili outlier customer: 2 righe (mostro le prime 10)


,CA_ID,city,Lat,Lon
71,60006377877,delhi,57.367808,154.320738
496,900001108756,mumbai,0.000000,0.000000


**Drop the two identified outliers**  


In [7]:
mask_outside_india = (~customer_data['Lat'].between(6, 38)) | (~customer_data['Lon'].between(68, 98))
mask_zero_zero     = (customer_data['Lat'].fillna(0) == 0) & (customer_data['Lon'].fillna(0) == 0)
customer_outliers  = customer_data.loc[mask_outside_india | mask_zero_zero, ['CA_ID','city','Lat','Lon']]


In [16]:
# Drop degli outlier
customer_data = customer_data.loc[~(mask_outside_india | mask_zero_zero)].copy()


In [ ]:
weather join

**Union of weather slices.**  
First we vertically append 2023 and 2024 for each city; then we append the two cities into a single, column‑consistent dataframe `weather_data`. This yields a global hourly panel across both geographies and years.

**Timestamp parsing.**  
I coerce the raw `timestamp` column(s) to `datetime64[ns]`, catching malformed strings with `errors='coerce'`. Downstream joins/interpolation rely on a clean, timezone‑aware index.

**Range/coverage diagnostics.**  
Compute per‑city minima, maxima, and counts to confirm temporal coverage and detect gaps/outliers, keeping the original values intact.


In [17]:
# Unions by city
weather_delhi  = pd.concat([weather23delhi,  weather24delhi],  ignore_index=True)
weather_mumbai = pd.concat([weather23mumbai, weather24mumbai], ignore_index=True)

# Final union
weather_data = pd.concat([weather_delhi, weather_mumbai], ignore_index=True).sort_values('timestamp').reset_index(drop=True)

# Parsing timestamp (robusto)
weather_data['timestamp'] = pd.to_datetime(weather_data['timestamp'], errors='coerce')

# Extract Lon/Lat from the 'location' column (format: POINT (lon lat))
if ('Lon' not in weather_data.columns) or ('Lat' not in weather_data.columns):
    ll = weather_data['location'].astype(str).str.extract(r'POINT\s*\(\s*([-\d\.]+)\s+([-\d\.]+)\s*\)')
    weather_data['Lon'] = pd.to_numeric(ll[0], errors='coerce')
    weather_data['Lat'] = pd.to_numeric(ll[1], errors='coerce')

# Numerical types for other weather measurements where possible
for c in weather_data.columns:
    if c not in ['timestamp','city','location','Lat','Lon'] and weather_data[c].dtype == 'object':
        weather_data[c] = pd.to_numeric(weather_data[c], errors='ignore')

print(f"[weather_data] shape: {weather_data.shape[0]:,} × {weather_data.shape[1]}")
print("Colonne:", list(weather_data.columns))

# NaN %
nan_pct_weather = (weather_data.isna().mean() * 100).round(2).sort_values(ascending=False)
print("\nNaN % (prime 15):")
print(nan_pct_weather.head(15))

# Overall time range and by city
print("\nRange temporale globale:")
print(weather_data['timestamp'].min(), "→", weather_data['timestamp'].max())

print("\nRange temporale per città:")
print(weather_data.groupby('city')['timestamp'].agg(['min','max','count']))


[weather_data] shape: 756,585 × 11
Colonne: ['timestamp', 'wind_speed', 'wind_direction', 'temperature', 'city', 'precipitation', 'location', 'surface_net_solar_radiation', 'surface_solar_radiation_downwards', 'Lon', 'Lat']

NaN % (prime 15):
surface_net_solar_radiation          12.31
surface_solar_radiation_downwards    12.31
timestamp                             0.00
wind_speed                            0.00
wind_direction                        0.00
temperature                           0.00
city                                  0.00
precipitation                         0.00
location                              0.00
Lon                                   0.00
Lat                                   0.00
dtype: float64

Range temporale globale:
2023-01-01 01:00:00 → 2024-12-31 23:00:00

Range temporale per città:
                       min                 max   count
city                                                  
delhi  2023-01-01 01:00:00 2024-12-31 23:00:00  504390
mumbai 2

In [11]:
print('weather unique locations:', weather_data.dropna(subset=['Lat','Lon'])[['city','Lat','Lon']].drop_duplicates().shape[0])

weather unique locations: 45


In [18]:
# Report gap rispetto alla frequenza più comune (di solito 1h)
def _gap_report(df, group_cols):
    rows = []
    for key, g in df.groupby(group_cols, dropna=False):
        s = g['timestamp'].sort_values().dropna()
        diffs = s.diff().dropna()
        if len(diffs) == 0:
            expected = pd.NaT
            n_gaps = 0
            max_gap = pd.Timedelta(0)
        else:
            expected = diffs.mode().iloc[0]          # frequenza attesa
            gaps = diffs[diffs > expected * 1.5]     # gap > 1.5× atteso
            n_gaps = int(gaps.shape[0])
            max_gap = gaps.max() if n_gaps else pd.Timedelta(0)
        key = (key,) if not isinstance(key, tuple) else key
        rows.append((*key, expected, n_gaps, max_gap))
    return pd.DataFrame(rows, columns=group_cols + ['freq_attesa','n_gaps','max_gap'])

print("\n=== GAP orari per città ===")
gap_city = _gap_report(weather_data, ['city'])
gap_city

print("\n=== GAP orari per (city, Lon, Lat) — top 10 più gravi ===")
gap_grid = _gap_report(weather_data, ['city','Lon','Lat']).sort_values(['n_gaps','max_gap'], ascending=[False, False]).head(10)
gap_grid



=== GAP orari per città ===

=== GAP orari per (city, Lon, Lat) — top 10 più gravi ===


,city,Lon,Lat,freq_attesa,n_gaps,max_gap
0,delhi,76.84,28.4,0 days 01:00:00,730,0 days 02:00:00
1,delhi,76.84,28.5,0 days 01:00:00,730,0 days 02:00:00
2,delhi,76.84,28.6,0 days 01:00:00,730,0 days 02:00:00
3,delhi,76.84,28.7,0 days 01:00:00,730,0 days 02:00:00
4,delhi,76.84,28.8,0 days 01:00:00,730,0 days 02:00:00
5,delhi,76.94,28.4,0 days 01:00:00,730,0 days 02:00:00
6,delhi,76.94,28.5,0 days 01:00:00,730,0 days 02:00:00
7,delhi,76.94,28.6,0 days 01:00:00,730,0 days 02:00:00
8,delhi,76.94,28.7,0 days 01:00:00,730,0 days 02:00:00
9,delhi,76.94,28.8,0 days 01:00:00,730,0 days 02:00:00


In [31]:
weather_data.to_pickle("data/weather_data.pkl")

In [19]:
present_by_city_hour = (
    weather_data.assign(hour=weather_data['timestamp'].dt.hour)
                .groupby(['city','hour']).size()
                .unstack(fill_value=0)
                .reindex(columns=range(24), fill_value=0)  # <-- evita il KeyError
)
display(present_by_city_hour)


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
city,,,,,,,,,,,,,,,,,,,,,
delhi,0,21930,21930,21930,21930,21930,21930,21930,21930,21930,...,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930
mumbai,0,10965,10965,10965,10965,10965,10965,10965,10965,10965,...,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965


In [30]:
# # Conteggi per ora (ora 0 ora c'è)
# present_by_city_hour_after = (
#     weather_filled.assign(hour=weather_filled['timestamp'].dt.hour)
#                  .groupby(['city','hour']).size()
#                  .unstack(fill_value=0)
#                  .reindex(columns=range(24), fill_value=0)
# )
# display(present_by_city_hour_after)

# # Gap per punto (dovrebbero essere 0)
# def gap_report_per_point(df):
#     rows = []
#     for (city, lon, lat), g in df.groupby(['city','Lon','Lat'], sort=False):
#         s = g['timestamp'].dropna().sort_values().unique()
#         diffs = pd.Series(s).diff().dropna()
#         if len(diffs) == 0:
#             expected = pd.Timedelta('1H'); n_gaps = 0; max_gap = pd.Timedelta(0)
#         else:
#             expected = diffs.mode().iloc[0]
#             gaps = diffs[diffs > expected * 1.5]
#             n_gaps = int(gaps.shape[0]); max_gap = gaps.max() if n_gaps else pd.Timedelta(0)
#         rows.append([city, lon, lat, expected, n_gaps, max_gap])
#     return pd.DataFrame(rows, columns=['city','Lon','Lat','freq_attesa','n_gaps','max_gap'])

# gap_points = gap_report_per_point(weather_filled)
# display(gap_points.groupby('city')['n_gaps'].sum())  # → 0 se tutto ok

# # NaN% prima/dopo (le due colonne solari devono scendere da ~12.31% a 0)
# nan_before = (weather_data.isna().mean()*100).round(2)
# nan_after  = (weather_filled.isna().mean()*100).round(2)
# print("NaN% prima:"); display(nan_before.sort_values(ascending=False))
# print("NaN% dopo:");  display(nan_after.sort_values(ascending=False))


hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
city,,,,,,,,,,,,,,,,,,,,,,,,
delhi,21900,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930
mumbai,10950,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965


city
delhi     0
mumbai    0
Name: n_gaps, dtype: int64

NaN% prima:


surface_net_solar_radiation          12.31
surface_solar_radiation_downwards    12.31
timestamp                             0.00
wind_speed                            0.00
wind_direction                        0.00
temperature                           0.00
city                                  0.00
precipitation                         0.00
location                              0.00
Lon                                   0.00
Lat                                   0.00
dtype: float64

NaN% dopo:


location                             4.16
timestamp                            0.00
wind_speed                           0.00
wind_direction                       0.00
temperature                          0.00
city                                 0.00
precipitation                        0.00
surface_net_solar_radiation          0.00
surface_solar_radiation_downwards    0.00
Lon                                  0.00
Lat                                  0.00
year                                 0.00
month                                0.00
hour                                 0.00
dtype: float64

In [20]:
wd = weather_data.sort_values(['city','Lon','Lat','timestamp']).copy()
wd['timestamp'] = pd.to_datetime(wd['timestamp'], errors='coerce')

full_parts = []
for (city, lon, lat), g in wd.groupby(['city','Lon','Lat'], sort=False):
    g = g.set_index('timestamp').sort_index()
    # Inizio: 00:00 del primo giorno; Fine: 23:00 dell’ultimo giorno
    start = g.index.min().normalize()
    end   = g.index.max().normalize() + pd.Timedelta(hours=23)
    idx = pd.date_range(start, end, freq='1h')

    gg = g.reindex(idx)
    # Propaga coordinate e city (costanti per il punto)
    gg[['city','Lon','Lat']] = gg[['city','Lon','Lat']].ffill().bfill()
    gg.index.name = 'timestamp'
    full_parts.append(gg.reset_index())

weather_full = (pd.concat(full_parts, ignore_index=True)
                .sort_values(['city','Lon','Lat','timestamp'])
                .reset_index(drop=True))

In [21]:
# Assicurati che Lon/Lat siano float e non NaN
assert weather_full['Lon'].notna().all() and weather_full['Lat'].notna().all()
weather_full['Lon'] = weather_full['Lon'].astype(float)
weather_full['Lat'] = weather_full['Lat'].astype(float)

# Ensure that Lon/Lat are floats and not NaN.
weather_full['location'] = ("POINT (" 
                            + weather_full['Lon'].round(6).astype(str) + " " 
                            + weather_full['Lat'].round(6).astype(str) + ")")


In [22]:
# support columns
weather_full['year']  = weather_full['timestamp'].dt.year
weather_full['month'] = weather_full['timestamp'].dt.month
weather_full['hour']  = weather_full['timestamp'].dt.hour

# --- Wind direction (circular)holes) ---
interp_parts = []
for key, g in weather_full.groupby(['city','Lon','Lat'], sort=False):
    g = g.set_index('timestamp').sort_index()
    # Direzione del vento (circolare)
    if 'wind_direction' in g.columns:
        ang = np.deg2rad(g['wind_direction'])
        sin = pd.Series(np.sin(ang), index=g.index).interpolate('time', limit=6, limit_direction='both')
        cos = pd.Series(np.cos(ang), index=g.index).interpolate('time', limit=6, limit_direction='both')
        g['wind_direction'] = (np.degrees(np.arctan2(sin, cos)) + 360) % 360

    # Other numbers
    for col in ['temperature','wind_speed','precipitation',
                'surface_solar_radiation_downwards','surface_net_solar_radiation']:
        if col in g:
            g[col] = g[col].interpolate('time', limit=6, limit_direction='both')

    # Physical constraints
    if 'wind_speed' in g: g['wind_speed'] = g['wind_speed'].clip(lower=0)
    if 'precipitation' in g: g['precipitation'] = g['precipitation'].clip(lower=0)

    interp_parts.append(g.reset_index())

weather_interp = (pd.concat(interp_parts, ignore_index=True)
                  .sort_values(['city','Lon','Lat','timestamp']))


In [23]:
rad_cols = ['surface_solar_radiation_downwards', 'surface_net_solar_radiation']
base = weather_interp[weather_interp['year'] == 2024]

clim_loc  = base.groupby(['city','Lon','Lat','month','hour'])[rad_cols].median().reset_index()
clim_city = base.groupby(['city','month','hour'])[rad_cols].median().reset_index()

filled = weather_interp.copy()
for col in rad_cols:
    m = filled[col].isna()
    if m.any():
        tmp = filled.loc[m, ['city','Lon','Lat','month','hour']].merge(
            clim_loc[['city','Lon','Lat','month','hour',col]],
            on=['city','Lon','Lat','month','hour'], how='left')[col]
        filled.loc[m, col] = tmp.values
    m = filled[col].isna()
    if m.any():
        tmp = filled.loc[m, ['city','month','hour']].merge(
            clim_city[['city','month','hour',col]],
            on=['city','month','hour'], how='left')[col]
        filled.loc[m, col] = tmp.values

    # Midnight = 0 + non-negativity
    filled.loc[filled['hour'] == 0, col] = 0.0
    filled[col] = filled[col].clip(lower=0)

# net ≤ downwards
if set(rad_cols).issubset(filled.columns):
    filled['surface_net_solar_radiation'] = np.minimum(
        filled['surface_net_solar_radiation'],
        filled['surface_solar_radiation_downwards']
    )

weather_filled = filled.sort_values(['city','Lon','Lat','timestamp']).reset_index(drop=True)


In [24]:
# Counts per hour (hour 0 must align with the others)
hour_counts = (weather_filled.assign(hour=weather_filled['timestamp'].dt.hour)
               .groupby(['city','hour']).size()
               .unstack(fill_value=0)
               .reindex(columns=range(24), fill_value=0))
display(hour_counts)

# NaN %: location deve essere 0.0
nan_after = (weather_filled.isna().mean()*100).round(2).sort_values(ascending=False)
display(nan_after.head(15))


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
city,,,,,,,,,,,,,,,,,,,,,
delhi,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930,...,21930,21930,21930,21930,21930,21930,21930,21930,21930,21930
mumbai,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965,...,10965,10965,10965,10965,10965,10965,10965,10965,10965,10965


timestamp                            0.0
wind_speed                           0.0
wind_direction                       0.0
temperature                          0.0
city                                 0.0
precipitation                        0.0
location                             0.0
surface_net_solar_radiation          0.0
surface_solar_radiation_downwards    0.0
Lon                                  0.0
Lat                                  0.0
year                                 0.0
month                                0.0
hour                                 0.0
dtype: float64

arrotondamento

In [ ]:
# weather_filled['Lon_round1'] = weather_filled['Lon'].round(1)
# weather_filled['Lat_round1'] = weather_filled['Lat'].round(1)


In [25]:
# lavorando su weather_filled, lo rendiamo canonico:
weather_data = weather_filled


In [4]:
weather_data.to_pickle("data/weather_data.pkl")

NameError: name 'weather_data' is not defined

In [4]:
weather_data = pd.read_pickle("data/weather_data.pkl")

In [26]:
weather_data.head()

,timestamp,wind_speed,wind_direction,temperature,city,precipitation,location,surface_net_solar_radiation,surface_solar_radiation_downwards,Lon,Lat,year,month,hour
0,2023-01-01 00:00:00,4.559800,88.316491,7.442108,delhi,0.0,POINT (76.84 28.4),0.000000,0.000000e+00,76.84,28.4,2023,1,0
1,2023-01-01 01:00:00,4.559800,88.316491,7.442108,delhi,0.0,POINT (76.84 28.4),0.000000,0.000000e+00,76.84,28.4,2023,1,1
2,2023-01-01 02:00:00,4.324242,78.425857,7.101471,delhi,0.0,POINT (76.84 28.4),3767.316162,4.683422e+03,76.84,28.4,2023,1,2
3,2023-01-01 03:00:00,2.914515,80.066128,8.798676,delhi,0.0,POINT (76.84 28.4),260158.265625,3.231988e+05,76.84,28.4,2023,1,3
4,2023-01-01 04:00:00,2.726061,84.380669,12.657867,delhi,0.0,POINT (76.84 28.4),990657.375000,1.230246e+06,76.84,28.4,2023,1,4


In [ ]:
customer powbal merge

In [27]:

# Working copies
df_powbal = df_switch_transformed.copy()
cust      = customer_data.copy()

# 1) Normalises keys (removes ".0" and invisible/zero-width spaces)
df_powbal['ca_number'] = (
    df_powbal['ca_number'].astype(str)
      .str.replace(r'\.0$', '', regex=True)
      .str.replace(r'[\u00A0\u200B]', '', regex=True)
      .str.strip()
)
cust['CA_ID'] = (
    cust['CA_ID'].astype(str)
      .str.replace(r'[\u00A0\u200B]', '', regex=True)
      .str.strip()
)

# 2) Lightweight mapping (no giant merges)
cust_idx = cust.set_index('CA_ID')[['Lat','Lon','city']]

powbal_customer = df_powbal.copy()
powbal_customer['Lat']  = pd.to_numeric(powbal_customer['ca_number'].map(cust_idx['Lat']), errors='coerce')
powbal_customer['Lon']  = pd.to_numeric(powbal_customer['ca_number'].map(cust_idx['Lon']), errors='coerce')
powbal_customer['city'] = powbal_customer['ca_number'].map(cust_idx['city']).astype('string').str.lower()

# 3) Coverage – unique rows and IDs (why do you see NaN?)
row_cov = powbal_customer[['Lat','Lon']].notna().all(axis=1).mean()
id_cov  = (powbal_customer.loc[powbal_customer[['Lat','Lon']].notna().all(axis=1), 'ca_number'].nunique()
           / powbal_customer['ca_number'].nunique())
print(f"Copertura righe: {row_cov:.2%}")
print(f"Copertura per ID unici: {id_cov:.2%}")

# How many IDs does each dataset have and how many do they have in common?
p_ids = pd.Index(powbal_customer['ca_number'].unique())
c_ids = pd.Index(cust['CA_ID'].unique())
inter = p_ids.intersection(c_ids)
print(f"ID unici powbal: {p_ids.size:,} | ID unici customer: {c_ids.size:,} | Intersezione: {inter.size:,}")

# 10 powbal IDs that are NOT in the customer (and therefore give NaN)
missing_ids = p_ids.difference(c_ids)[:10]
print("ID non presenti nel customer (prime 10):")
display(pd.DataFrame({'ca_number_missing': missing_ids}))


Copertura righe: 58.34%
Copertura per ID unici: 48.59%
ID unici powbal: 992 | ID unici customer: 512 | Intersezione: 482
ID non presenti nel customer (prime 10):


,ca_number_missing
0,60000191183
1,60000254585
2,60000318224
3,60000319917
4,60000333272
5,60000334361
6,60000335624
7,60000340327
8,60000501258
9,60000550966


In [14]:
powbal_customer["Lat"].unique()

array([        nan, 28.69557317, 28.69549906, 28.62371412, 28.62732372,
       28.71146779, 28.68811523, 28.64900739, 28.70367023, 28.72341146,
       28.68158383, 28.68455404, 28.70359063, 28.69401626, 28.70115624,
       28.69169494, 28.71413005, 28.71128974, 28.70369197, 28.73575226,
       28.70960906, 28.69296832, 28.69215144, 28.80335505, 28.66817851,
       28.68507068, 28.69527847, 28.69404879, 28.68100549, 28.70674055,
       28.83781801, 28.70495765, 28.68237222, 28.6938432 , 28.73900204,
       28.6949464 , 28.68109956, 28.69675828, 28.69340867, 28.70186248,
       28.7059134 , 28.68738111, 28.69449999, 28.70660213, 28.71359089,
       28.71467536, 28.75065991, 28.70572328, 28.69417468, 28.67523572,
       28.68072814, 28.73045745, 28.69735079, 28.68248918, 28.68282625,
       28.70741189, 28.69705191, 28.69150398, 28.68215823, 28.69762271,
       28.68589162, 28.68124004, 28.70861915, 28.7060866 , 28.7461369 ,
       28.68992691, 28.68523349, 28.692265  , 28.67039735, 28.72

In [15]:
powbal_customer.head()

,power,schedule_id,ca_number,ca_number_str,timestamp,off,on,reward_rate,notify_participant_at,posted_to_api,...,evening_share,night_share,pod_entropy,excess_weekend_share,peak_energy_share,circadian_concentration,day_to_day_cv,peakiness_p95,Lat,Lon
0,66.309998,NaN,60000191183,60000191183,2023-10-17 04:00,NaN,NaN,0,NaN,NaN,...,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971,NaN,NaN
1,70.660004,NaN,60000191183,60000191183,2023-10-17 04:31,NaN,NaN,0,NaN,NaN,...,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971,NaN,NaN
2,28.690001,NaN,60000191183,60000191183,2023-10-17 06:00,NaN,NaN,0,NaN,NaN,...,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971,NaN,NaN
3,29.820000,NaN,60000191183,60000191183,2023-10-17 07:00,NaN,NaN,0,NaN,NaN,...,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971,NaN,NaN
4,79.440002,NaN,60000191183,60000191183,2023-10-17 07:30,NaN,NaN,0,NaN,NaN,...,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971,NaN,NaN


In [18]:
powbal_customer["city"].unique()

<StringArray>
[<NA>, 'delhi', 'mumbai']
Length: 3, dtype: string

In [28]:

# Inputs:
# df_powbal         -> POWBAL with columns ['ca_number','city', <time col> ...]
# customer_data     -> unique ['CA_ID','Lat','Lon']  (outliers already dropped)
# weather_data      -> ['city','timestamp','Lat','Lon', <weather columns>]

# 1) Normalize keys (remove trailing ".0", invisible spaces)
df_powbal = df_powbal.copy()
customer  = customer_data.copy()

df_powbal['ca_number'] = (df_powbal['ca_number'].astype(str)
                          .str.replace(r'\.0$', '', regex=True)
                          .str.replace(r'[\u00A0\u200B]', '', regex=True)
                          .str.strip())

customer['CA_ID'] = (customer['CA_ID'].astype(str)
                     .str.replace(r'[\u00A0\u200B]', '', regex=True)
                     .str.strip())

# 2) Map Lat/Lon by ID (LEFT semantics). Keep POWBAL 'city' as authoritative.
lat_map = customer.set_index('CA_ID')['Lat']
lon_map = customer.set_index('CA_ID')['Lon']

powbal = df_powbal.copy()
powbal['city'] = powbal['city'].astype(str).str.strip().str.lower()
powbal['Lat']  = pd.to_numeric(powbal['ca_number'].map(lat_map), errors='coerce')
powbal['Lon']  = pd.to_numeric(powbal['ca_number'].map(lon_map), errors='coerce')

print("Row‑coverage after ID mapping:",
      powbal[['Lat','Lon']].notna().all(axis=1).mean())
print("Unique ID coverage:",
      (powbal.loc[powbal[['Lat','Lon']].notna().all(axis=1),'ca_number'].nunique()
       / powbal['ca_number'].nunique()))


Row‑coverage after ID mapping: 0.5834301935034166
Unique ID coverage: 0.48588709677419356


**Persist results.**  
We save intermediate/final artifacts to disk (`pickle/Parquet/CSV`) to support reproducibility and incremental processing on HPC. Row‑grouping is tuned to keep memory stable at write time.

> **Why:** clean, well-documented preprocessing makes the downstream LSTM feature ingestion deterministic and auditable.

In [12]:
powbal.to_pickle("data/powbal.pkl")

In [4]:
powbal = pd.read_pickle("data/powbal.pkl")

In [ ]:
weather_data = pd.read_pickle("data/weather_data.pkl")

In [19]:
powbal["city"].unique()

array(['delhi', 'mumbai'], dtype=object)

In [20]:
powbal["Lat"].unique()

array([        nan, 28.69557317, 28.69549906, 28.62371412, 28.62732372,
       28.71146779, 28.68811523, 28.64900739, 28.70367023, 28.72341146,
       28.68158383, 28.68455404, 28.70359063, 28.69401626, 28.70115624,
       28.69169494, 28.71413005, 28.71128974, 28.70369197, 28.73575226,
       28.70960906, 28.69296832, 28.69215144, 28.80335505, 28.66817851,
       28.68507068, 28.69527847, 28.69404879, 28.68100549, 28.70674055,
       28.83781801, 28.70495765, 28.68237222, 28.6938432 , 28.73900204,
       28.6949464 , 28.68109956, 28.69675828, 28.69340867, 28.70186248,
       28.7059134 , 28.68738111, 28.69449999, 28.70660213, 28.71359089,
       28.71467536, 28.75065991, 28.70572328, 28.69417468, 28.67523572,
       28.68072814, 28.73045745, 28.69735079, 28.68248918, 28.68282625,
       28.70741189, 28.69705191, 28.69150398, 28.68215823, 28.69762271,
       28.68589162, 28.68124004, 28.70861915, 28.7060866 , 28.7461369 ,
       28.68992691, 28.68523349, 28.692265  , 28.67039735, 28.72

In [29]:
powbal_ns = powbal  # alias, so all IDW code that uses powbal_ns remains valid


**Timestamp parsing.**  
We coerce the raw `timestamp` column(s) to `datetime64[ns]`, catching malformed strings with `errors='coerce'`. Downstream joins/interpolation rely on a clean, timezone‑aware index.

**Time‑zone normalization.**  
POWBAL events are handled in *Asia/Kolkata*, whereas some weather sources are already in UTC. We localize if naive, otherwise we convert to UTC and preserve an aligned `timestamp_utc` that is used in temporal matching.

> **Why:** clean, well-documented preprocessing makes the downstream LSTM feature ingestion deterministic and auditable.

In [ ]:
POWBAL_TZ  = 'Asia/Kolkata'   # POWBAL timezone
WEATHER_TZ = 'Asia/Kolkata'   # set to 'UTC' if weather timestamps already in UTC
ts_col     = 'parsed_datetime' # your best POWBAL time column

def to_utc(series, tz_hint: str) -> pd.Series:
    s = pd.to_datetime(series, errors='coerce')
    try:
        if s.dt.tz is not None:
            return s.dt.tz_convert('UTC')
    except Exception:
        pass
    if tz_hint.upper() == 'UTC':
        return s.dt.tz_localize('UTC', nonexistent='shift_forward', ambiguous='NaT')
    return s.dt.tz_localize(tz_hint, nonexistent='shift_forward', ambiguous='NaT').dt.tz_convert('UTC')

powbal_ns  = powbal.copy()
weather = weather_data.copy()

powbal_ns['timestamp_utc']  = to_utc(powbal[ts_col], POWBAL_TZ)
weather['timestamp_utc'] = to_utc(weather['timestamp'], WEATHER_TZ)
weather['city']          = weather['city'].astype(str).str.strip().str.lower()


In [ ]:
2

In [8]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# ---------- config ----------
K = 4
P = 2.0
EPS = 1e-6
wx_cols = [
    'temperature','wind_speed','wind_direction','precipitation',
    'surface_net_solar_radiation','surface_solar_radiation_downwards'
]
loc_cols = ['city','Lat','Lon']

# ---------- helpers ----------
def to_rad(x):  # deg -> rad
    return np.deg2rad(x.astype(np.float64))

def temporal_interp_arrays(times, right_df, cols):
    """Linear (circular for wind_direction) interpolation at 'times'."""
    left  = pd.DataFrame({'timestamp_utc': pd.Index(times)}).sort_values('timestamp_utc').reset_index(drop=True)
    right = right_df[['timestamp_utc'] + cols].sort_values('timestamp_utc').reset_index(drop=True)
    if right.empty:
        return {c: np.full(len(left), np.nan) for c in cols}

    prev_ = pd.merge_asof(left, right, on='timestamp_utc', direction='backward', allow_exact_matches=True)
    prev_.columns = ['timestamp_utc'] + [f'{c}_prev' for c in cols]
    next_ = pd.merge_asof(left, right, on='timestamp_utc', direction='forward', allow_exact_matches=False)
    next_.columns = ['timestamp_utc'] + [f'{c}_next' for c in cols]
    tmp = prev_.merge(next_, on='timestamp_utc', how='left')

    rt  = right[['timestamp_utc']].rename(columns={'timestamp_utc':'t'})
    lft = left[['timestamp_utc']].rename(columns={'timestamp_utc':'t'})
    p_times = pd.merge_asof(lft, rt, on='t', direction='backward')['t']
    n_times = pd.merge_asof(lft, rt, on='t', direction='forward')['t']
    dt_sec = (n_times - p_times).dt.total_seconds().to_numpy()
    w_sec  = (left['timestamp_utc'] - p_times).dt.total_seconds().to_numpy()
    alpha  = np.zeros(len(left), dtype='float64')
    ok     = np.isfinite(dt_sec) & (dt_sec > 0)
    alpha[ok] = np.clip(w_sec[ok] / dt_sec[ok], 0, 1)

    out = {}
    for col in cols:
        a = tmp[f'{col}_prev'].to_numpy()
        b = tmp[f'{col}_next'].to_numpy()
        if col == 'wind_direction':
            ar = np.deg2rad(a); br = np.deg2rad(b)
            sin_i = np.where(~np.isnan(b), (1-alpha)*np.sin(ar) + alpha*np.sin(br), np.sin(ar))
            cos_i = np.where(~np.isnan(b), (1-alpha)*np.cos(ar) + alpha*np.cos(br), np.cos(ar))
            use_next = np.isnan(a) & ~np.isnan(b)
            sin_i = np.where(use_next, np.sin(br), sin_i)
            cos_i = np.where(use_next, np.cos(br), cos_i)
            arr = (np.degrees(np.arctan2(sin_i, cos_i)) + 360) % 360
        else:
            arr = np.where(~np.isnan(b), (1-alpha)*a + alpha*b, a)
            arr = np.where(np.isnan(a), b, arr)
            if col in ['precipitation','surface_net_solar_radiation',
                       'surface_solar_radiation_downwards','wind_speed']:
                arr = np.clip(arr, 0, None)
        out[col] = arr
    return out

# ---------- build per-city grid & BallTree ----------
weather_data['city'] = weather_data['city'].astype(str).str.strip().str.lower()
powbal_ns['city']    = powbal_ns['city'].astype(str).str.strip().str.lower()

wx_grid = (weather[['city','Lat','Lon']]
           .drop_duplicates()
           .reset_index(drop=True))

trees, grids = {}, {}
for city, g in wx_grid.groupby('city', observed=True):
    grids[city] = g.reset_index(drop=True)
    trees[city] = BallTree(to_rad(grids[city][['Lat','Lon']].to_numpy()), metric='haversine')

# ---------- precompute temporal interpolation per node & city ----------
node_cache = {}   # {city: {node_idx: {col: array(len(T_city))}}}
time_index = {}   # {city: np.ndarray T_city}

for city, dfc in powbal_ns.groupby('city', observed=True):
    T_city = np.sort(dfc['timestamp_utc'].dropna().unique())
    time_index[city] = T_city
    node_cache[city] = {}

    for j, row in grids[city].iterrows():
        latj, lonj = float(row['Lat']), float(row['Lon'])
        right = weather[(weather['city']==city) &
                             (weather['Lat']==latj) &
                             (weather['Lon']==lonj)]
        node_cache[city][j] = temporal_interp_arrays(T_city, right, wx_cols)

# ---------- IDW by location (k=4) using precomputed nodes ----------
pieces = []
for (city, la, lo), g in powbal_ns.groupby(['city','Lat','Lon'], observed=True):
    if city not in trees: 
        continue
    T_city = time_index[city]
    times  = np.sort(g['timestamp_utc'].dropna().unique())
    # posizioni di times all'interno di T_city
    pos = pd.Index(T_city).get_indexer(times)

    # k vicini
    G = grids[city]
    tree = trees[city]
    dist_rad, idx = tree.query(to_rad(np.array([[la, lo]])), k=min(K, len(G)))
    idx = idx.ravel()
    dist_km = (dist_rad.ravel() * 6371.0088).astype('float64')

    # se un nodo coincide (dist < 0.1km) usa solo quello (niente IDW)
    near = np.argmin(dist_km)
    if dist_km[near] < 0.1:
        idx = np.array([idx[near]])
        dist_km = dist_km[[near]]

    w = 1.0 / np.maximum(dist_km, EPS)**P
    w = w / w.sum()

    # accumulate weight
    M = len(times)
    num = {c: np.zeros(M, dtype='float64') for c in wx_cols if c != 'wind_direction'}
    den = {c: np.zeros(M, dtype='float64') for c in wx_cols if c != 'wind_direction'}
    sin_num = np.zeros(M, dtype='float64'); cos_num = np.zeros(M, dtype='float64'); den_wd = np.zeros(M, dtype='float64')

    for j, wj in zip(idx, w):
        arrs = node_cache[city][int(j)]
        for col in wx_cols:
            full = arrs[col]
            sel  = full[pos]  # seleziona i tempi richiesti
            m    = ~np.isnan(sel)
            if col == 'wind_direction':
                sin_num[m] += wj * np.sin(np.deg2rad(sel[m]))
                cos_num[m] += wj * np.cos(np.deg2rad(sel[m]))
                den_wd[m]  += wj
            else:
                num[col][m] += wj * sel[m]
                den[col][m] += wj

    out = pd.DataFrame({'timestamp_utc': times, 'city': city, 'Lat': la, 'Lon': lo})
    for col in wx_cols:
        if col == 'wind_direction':
            wd = np.degrees(np.arctan2(sin_num, cos_num))
            wd = (wd + 360) % 360
            wd = np.where(den_wd > 0, wd, np.nan)
            out[col] = wd.astype('float32')
        else:
            val = np.divide(num[col], den[col],
                            out=np.full(M, np.nan, dtype='float64'),
                            where=den[col] > 0)
            if col in ['precipitation','surface_net_solar_radiation','surface_solar_radiation_downwards','wind_speed']:
                val = np.clip(val, 0, None)
            out[col] = val.astype('float32')

    pieces.append(out)

wx_at_powbal_times_idw = pd.concat(pieces, ignore_index=True)
print("IDW rows:", len(wx_at_powbal_times_idw))


IDW rows: 4312272


In [10]:
wx_at_powbal_times_idw.to_pickle("data/wx_at_powbal_times.pkl")

In [3]:
wx_at_powbal_times_idw = pd.read_pickle("data/wx_at_powbal_times.pkl")

In [9]:
wx_at_powbal_times_idw.head()

,timestamp_utc,city,Lat,Lon,temperature,wind_speed,wind_direction,precipitation,surface_net_solar_radiation,surface_solar_radiation_downwards
0,2023-08-11 14:30:00+00:00,delhi,28.623714,77.137896,28.817507,6.562887,64.105438,0.000047,16259081.0,20132704.0
1,2023-08-11 15:00:00+00:00,delhi,28.623714,77.137896,28.817507,6.562887,64.105438,0.000047,16259081.0,20132704.0
2,2023-08-11 15:30:00+00:00,delhi,28.623714,77.137896,28.719734,8.239905,76.785469,0.000047,16259081.0,20132704.0
3,2023-08-11 16:00:00+00:00,delhi,28.623714,77.137896,28.719734,8.239905,76.785469,0.000047,16259081.0,20132704.0
4,2023-08-11 16:30:00+00:00,delhi,28.623714,77.137896,28.622850,9.189094,85.969131,0.000047,16259081.0,20132704.0


In [ ]:
3

In [10]:
merge_keys = ['city','Lat','Lon','timestamp_utc']
wx_cols_idw = [c for c in wx_at_powbal_times_idw.columns if c not in merge_keys]

# opzionale: assicurati che il lato meteo sia univoco sulle chiavi
dups = wx_at_powbal_times_idw.duplicated(subset=merge_keys, keep=False)
if dups.any():
    wx_at_powbal_times_idw = wx_at_powbal_times_idw.groupby(merge_keys, as_index=False)[wx_cols_idw].mean()

powbal_weather_idw = powbal_ns.merge(
    wx_at_powbal_times_idw, on=merge_keys, how='left', sort=False, validate='many_to_one'
)

print("locations POWBAL:", powbal_ns[loc_cols].drop_duplicates().shape[0],
      "-> after merge:", powbal_weather_idw[loc_cols].drop_duplicates().shape[0])
print("coverage:", powbal_weather_idw[wx_cols_idw].notna().any(axis=1).mean())


locations POWBAL: 484 -> after merge: 484
coverage: 0.5834301935034166


**Spatial handling (snap/IDW).**  
Where exact coordinate matches are not available, we either *snap* coordinates to the nearest weather grid point or use Inverse Distance Weighting over the k closest nodes. This preserves local realism without over‑fitting a single station.

> **Why:** clean, well-documented preprocessing makes the downstream LSTM feature ingestion deterministic and auditable.

In [12]:
powbal_weather_idw.head()

,power,schedule_id,ca_number,ca_number_str,timestamp,off,on,reward_rate,notify_participant_at,posted_to_api,...,peakiness_p95,Lat,Lon,timestamp_utc,temperature,wind_speed,wind_direction,precipitation,surface_net_solar_radiation,surface_solar_radiation_downwards
0,66.309998,NaN,60000191183,60000191183,2023-10-17 04:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
1,70.660004,NaN,60000191183,60000191183,2023-10-17 04:31,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 04:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2,28.690001,NaN,60000191183,60000191183,2023-10-17 06:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
3,29.820000,NaN,60000191183,60000191183,2023-10-17 07:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN
4,79.440002,NaN,60000191183,60000191183,2023-10-17 07:30,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 07:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
powbal_weather_idw["temperature"].unique()

array([      nan,  9.830232,  9.203483, ..., 25.694286, 25.91306 ,
       26.282143], dtype=float32)

In [ ]:
ULTIMO 4

In [11]:
from sklearn.neighbors import BallTree

# --- 0) Colonne meteo usate ---
wx_cols = [
    'temperature','wind_speed','wind_direction','precipitation',
    'surface_net_solar_radiation','surface_solar_radiation_downwards'
]

# --- 1) Used weather columnso città ---
if 'customer_data' in globals():
    src = customer_data[['city','Lat','Lon']].dropna(subset=['Lat','Lon']).copy()
    print("city_center source = customer_data")
elif 'powbal_ns' in globals() and {'city','Lat','Lon'}.issubset(powbal_ns.columns):
    src = powbal_ns[['city','Lat','Lon']].dropna(subset=['Lat','Lon']).copy()
    print("city_center source = powbal_ns (rows with Lat/Lon)")
else:
    src = weather[['city','Lat','Lon']].drop_duplicates().copy()
    print("city_center source = weather grid nodes (unique)")

src['city'] = src['city'].astype(str).str.strip().str.lower()
city_center = (src.groupby('city', as_index=False)[['Lat','Lon']]
               .median()
               .rename(columns={'Lat':'Lat_med','Lon':'Lon_med'}))

# --- 2) Build BallTree for city if missing ---
def to_rad(x): return np.deg2rad(x.astype(np.float64))

if 'trees' not in globals() or 'grids' not in globals():
    trees, grids = {}, {}
    wx_grid = weather[['city','Lat','Lon']].drop_duplicates().reset_index(drop=True)
    for city, g in wx_grid.groupby('city', observed=True):
        grids[city] = g.reset_index(drop=True)
        trees[city] = BallTree(to_rad(grids[city][['Lat','Lon']].to_numpy()), metric='haversine')

# Check: you need to have already done the temporal precompute for each node.
assert 'time_index' in globals() and 'node_cache' in globals(), \
    "Esegui prima il precompute temporale per nodo (time_index, node_cache)."

# --- 3) IDW at city level (city + timestamp) ---
K, P, EPS = 4, 2.0, 1e-6
rows = []

for _, rowc in city_center.iterrows():
    c   = str(rowc['city']).strip().lower()
    la0 = float(rowc['Lat_med'])
    lo0 = float(rowc['Lon_med'])

    if c not in trees or c not in time_index:
        continue

    T_city = time_index[c]                                # tempi precomputati per la città
    tree   = trees[c]
    grid   = grids[c]

    dist_rad, idx = tree.query(to_rad(np.array([[la0, lo0]])), k=min(K, len(grid)))
    idx = idx.ravel()
    dist_km = (dist_rad.ravel()*6371.0088).astype('float64')

    # if the centre is practically on a node (<100m), use only that one
    near = np.argmin(dist_km)
    if dist_km[near] < 0.1:
        idx = np.array([idx[near]])
        dist_km = dist_km[[near]]

    w = 1.0 / np.maximum(dist_km, EPS)**P
    w = w / w.sum()

    M = len(T_city)
    num = {col: np.zeros(M) for col in wx_cols if col != 'wind_direction'}
    den = {col: np.zeros(M) for col in wx_cols if col != 'wind_direction'}
    sin_num = np.zeros(M); cos_num = np.zeros(M); den_wd = np.zeros(M)

    for j, wj in zip(idx, w):
        arrs = node_cache[c][int(j)]                      # array temporali precomputati per nodo j
        for col in wx_cols:
            arr = arrs[col]
            m   = ~np.isnan(arr)
            if col == 'wind_direction':
                sin_num[m] += wj*np.sin(np.deg2rad(arr[m]))
                cos_num[m] += wj*np.cos(np.deg2rad(arr[m]))
                den_wd[m]  += wj
            else:
                num[col][m] += wj*arr[m]
                den[col][m] += wj

    out = pd.DataFrame({'city': c, 'timestamp_utc': pd.Index(T_city)})
    for col in wx_cols:
        if col == 'wind_direction':
            wd = np.degrees(np.arctan2(sin_num, cos_num))
            wd = (wd + 360) % 360
            wd = np.where(den_wd > 0, wd, np.nan)
            out[col] = wd.astype('float32')
        else:
            val = np.divide(num[col], den[col],
                            out=np.full(M, np.nan, dtype='float64'),
                            where=den[col] > 0)
            if col in ['precipitation','surface_net_solar_radiation',
                       'surface_solar_radiation_downwards','wind_speed']:
                val = np.clip(val, 0, None)
            out[col] = val.astype('float32')
    rows.append(out)

wx_city_time_idw = pd.concat(rows, ignore_index=True)

# --- 4) Fallback on rows still NaN after the "precision" merge ---
merge_keys = ['city','Lat','Lon','timestamp_utc']
wx_cols_idw = [c for c in wx_at_powbal_times_idw.columns if c not in merge_keys]

mask_na = ~powbal_weather_idw[wx_cols_idw].notna().any(axis=1)

# We only combine city+timestamp from NaN rows with the city IDW fallback.
fback = (powbal_weather_idw.loc[mask_na, ['city','timestamp_utc']]
         .merge(wx_city_time_idw, on=['city','timestamp_utc'], how='left'))

# fill in the weather columns where there were NaN values
for col in wx_cols_idw:
    powbal_weather_idw.loc[mask_na, col] = fback[col].values

print("Coverage after fallback:",
      powbal_weather_idw[wx_cols_idw].notna().any(axis=1).mean())


city_center source = powbal_ns (rows with Lat/Lon)
Coverage after fallback: 1.0


In [12]:
powbal_weather_idw.head()

,power,schedule_id,ca_number,ca_number_str,timestamp,off,on,reward_rate,notify_participant_at,posted_to_api,...,peakiness_p95,Lat,Lon,timestamp_utc,temperature,wind_speed,wind_direction,precipitation,surface_net_solar_radiation,surface_solar_radiation_downwards
0,66.309998,NaN,60000191183,60000191183,2023-10-17 04:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 04:00:00+00:00,26.468197,3.384724,218.257065,0.001025,9884670.0,11882028.0
1,70.660004,NaN,60000191183,60000191183,2023-10-17 04:31,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 04:30:00+00:00,26.605185,3.335279,217.966187,0.001044,10938983.0,13149465.0
2,28.690001,NaN,60000191183,60000191183,2023-10-17 06:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 06:00:00+00:00,26.593966,2.138674,214.075745,0.001044,11760603.0,14137130.0
3,29.820000,NaN,60000191183,60000191183,2023-10-17 07:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 07:00:00+00:00,25.971527,1.689554,191.422226,0.001044,12031999.0,14463382.0
4,79.440002,NaN,60000191183,60000191183,2023-10-17 07:30,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 07:30:00+00:00,25.070456,2.734958,191.229050,0.001044,12040863.0,14474043.0


In [13]:
powbal_weather_idw.to_pickle("data/powbal_weather.pkl")

In [2]:
powbal_weather = pd.read_pickle("data/powbal_weather.pkl")

In [3]:
powbal_weather["temperature"].unique()

array([26.468197, 26.605185, 26.593966, ..., 30.907146, 26.414888,
       30.664843], dtype=float32)

In [4]:
powbal_weather["precipitation"].unique()

array([0.00102507, 0.00104364, 0.0010445 , ..., 0.00105305, 0.00182135,
       0.00713632], dtype=float32)

In [3]:
powbal_weather.head()

,power,schedule_id,ca_number,ca_number_str,timestamp,off,on,reward_rate,notify_participant_at,posted_to_api,...,peakiness_p95,Lat,Lon,timestamp_utc,temperature,wind_speed,wind_direction,precipitation,surface_net_solar_radiation,surface_solar_radiation_downwards
0,66.309998,NaN,60000191183,60000191183,2023-10-17 04:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 04:00:00+00:00,26.468197,3.384724,218.257065,0.001025,9884670.0,11882028.0
1,70.660004,NaN,60000191183,60000191183,2023-10-17 04:31,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 04:30:00+00:00,26.605185,3.335279,217.966187,0.001044,10938983.0,13149465.0
2,28.690001,NaN,60000191183,60000191183,2023-10-17 06:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 06:00:00+00:00,26.593966,2.138674,214.075745,0.001044,11760603.0,14137130.0
3,29.820000,NaN,60000191183,60000191183,2023-10-17 07:00,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 07:00:00+00:00,25.971527,1.689554,191.422226,0.001044,12031999.0,14463382.0
4,79.440002,NaN,60000191183,60000191183,2023-10-17 07:30,NaN,NaN,0,NaN,NaN,...,0.478971,NaN,NaN,2023-10-17 07:30:00+00:00,25.070456,2.734958,191.229050,0.001044,12040863.0,14474043.0


In [6]:
df_switch_transformed.head()

,power,schedule_id,ca_number,ca_number_str,timestamp,off,on,reward_rate,notify_participant_at,posted_to_api,...,morning_share,afternoon_share,evening_share,night_share,pod_entropy,excess_weekend_share,peak_energy_share,circadian_concentration,day_to_day_cv,peakiness_p95
0,66.309998,NaN,60000191183,60000191183,2023-10-17 04:00,NaN,NaN,0,NaN,NaN,...,0.091345,0.292303,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971
1,70.660004,NaN,60000191183,60000191183,2023-10-17 04:31,NaN,NaN,0,NaN,NaN,...,0.091345,0.292303,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971
2,28.690001,NaN,60000191183,60000191183,2023-10-17 06:00,NaN,NaN,0,NaN,NaN,...,0.091345,0.292303,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971
3,29.820000,NaN,60000191183,60000191183,2023-10-17 07:00,NaN,NaN,0,NaN,NaN,...,0.091345,0.292303,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971
4,79.440002,NaN,60000191183,60000191183,2023-10-17 07:30,NaN,NaN,0,NaN,NaN,...,0.091345,0.292303,0.192052,0.12844,0.895004,0.000759,0.192052,0.150208,0.378456,0.478971


1

2

1?  powbal_ns

In [ ]:
# import pandas as pd
# import numpy as np

#     # Copie di lavoro
# df_powbal = df_switch_transformed.copy()
# cust      = customer_data.copy()

# # 2.1 Pulisci le chiavi (evita '... .0', spazi, NBSP/ZWSP)
# df_powbal['ca_number'] = (
#     df_powbal['ca_number'].astype(str)
#     .str.replace(r'\.0$', '', regex=True)            # rimuove .0 finale
#     .str.replace(r'[\u00A0\u200B]', '', regex=True)  # spazi invisibili
#     .str.strip()
# )
# cust['CA_ID'] = (
#     cust['CA_ID'].astype(str)
#     .str.replace(r'[\u00A0\u200B]', '', regex=True)
#     .str.strip()
# )

# # 2.2 Crea indici per mappare (nessun join gigantesco)
# cust_idx = cust.set_index('CA_ID')[['Lat','Lon','city']]

# # 2.3 Costruisci powbal_customer con map (veloce e parco RAM)
# powbal_customer = df_powbal.copy()

# # se powbal aveva già una 'city', salvala per non confonderti nel debug
# if 'city' in powbal_customer.columns:
#     powbal_customer = powbal_customer.rename(columns={'city':'city_pow'})

# powbal_customer['Lat']  = pd.to_numeric(powbal_customer['ca_number'].map(cust_idx['Lat']), errors='coerce')
# powbal_customer['Lon']  = pd.to_numeric(powbal_customer['ca_number'].map(cust_idx['Lon']), errors='coerce')
# powbal_customer['city'] = powbal_customer['ca_number'].map(cust_idx['city']).astype('string').str.lower()

# # 2.4 Diagnostica concisa (senza creare stringhe giganti)
# matched_rows = powbal_customer[['Lat','Lon']].notna().all(axis=1)
# print(f"Copertura coordinate dopo mapping: {matched_rows.mean():.2%}  "
#       f"({matched_rows.sum():,}/{len(powbal_customer):,})")

# unmatched_ids = (powbal_customer.loc[~matched_rows, 'ca_number']
#                  .drop_duplicates().head(10))
# print("ID senza match (prime 10):")
# display(unmatched_ids.to_frame('ca_number'))

# # (facoltativo) coverage per ID unici — utile per capire se il problema è diffuso
# id_cov = (powbal_customer.loc[matched_rows, 'ca_number'].nunique()
#           / powbal_customer['ca_number'].nunique())
# print(f"Copertura per ID unici: {id_cov:.2%}")


In [ ]:
# import pandas as pd
# import numpy as np

# # Copie di lavoro
# df_powbal = df_switch_transformed.copy()
# cust      = customer_data.copy()

# # 2.1 Normalizza le chiavi (tutto vettoriale, no apply)
# #    - rimuove ".0" di ca_number (se era float), spazi e char invisibili NBSP/ZWSP
# df_powbal['ca_number'] = (
#     df_powbal['ca_number']
#       .astype(str)
#       .str.replace(r'\.0$', '', regex=True)
#       .str.replace(r'[\u00A0\u200B]', '', regex=True)
#       .str.strip()
# )

# cust['CA_ID'] = (
#     cust['CA_ID']
#       .astype(str)
#       .str.replace(r'[\u00A0\u200B]', '', regex=True)
#       .str.strip()
# )

# # 2.2 Merge (m:1), suffix espliciti; prendi city dal customer
# cols_cust = ['CA_ID','Lat','Lon','city']
# powbal_customer = df_powbal.merge(
#     cust[cols_cust], how='left',
#     left_on='ca_number', right_on='CA_ID',
#     validate='m:1', suffixes=('', '_cust')
# ).drop(columns=['CA_ID'])

# if 'city_cust' in powbal_customer.columns:
#     powbal_customer['city'] = powbal_customer['city_cust']
#     powbal_customer.drop(columns=['city_cust'], inplace=True)

# # 2.3 Tipi coerenti + WKT (Lat/Lon restano numerici — nessun POINT qui)
# powbal_customer['Lat']  = pd.to_numeric(powbal_customer['Lat'], errors='coerce')
# powbal_customer['Lon']  = pd.to_numeric(powbal_customer['Lon'], errors='coerce')
# powbal_customer['city'] = powbal_customer['city'].astype(str).str.strip().str.lower()

# powbal_customer['location'] = (
#     "POINT (" + powbal_customer['Lon'].round(6).astype(str) + " " +
#                 powbal_customer['Lat'].round(6).astype(str) + ")"
# )

# # 2.4 Diagnostica ultra‑breve (copertura righe e 10 ID non matchati)
# matched = powbal_customer[['Lat','Lon']].notna().all(axis=1)
# print(f"Copertura coordinate dopo merge: {matched.mean():.2%}  ({matched.sum():,}/{len(powbal_customer):,})")

# unmatched_ids = (
#     powbal_customer.loc[~matched, 'ca_number']
#       .drop_duplicates()
#       .head(10)
# )
# print("ID senza match (prime 10):")
# display(unmatched_ids.to_frame('ca_number'))

# # (facoltativo) verifica immediata su un ID specifico che ti dava NaN
# # probe = '60000191183'
# # display(cust.loc[cust['CA_ID'] == probe, ['CA_ID','city','Lat','Lon']])
# # display(powbal_customer.loc[powbal_customer['ca_number'] == probe, ['ca_number','city','Lat','Lon']].head())


In [13]:
# # Assumo che esistano: df_switch_transformed, customer_data
# import pandas as pd
# import numpy as np

# # 2.1 Normalizza le chiavi come stringhe
# df_powbal = df_switch_transformed.copy()
# df_powbal['ca_number'] = df_powbal['ca_number'].astype(str).str.strip()
# customer_data['CA_ID'] = customer_data['CA_ID'].astype(str).str.strip()

# # 2.2 Seleziona solo le colonne geografiche da portare nel powbal
# cols_cust = ['CA_ID', 'Lat', 'Lon', 'city']
# powbal_customer = df_powbal.merge(customer_data[cols_cust],
#                                   how='left', left_on='ca_number', right_on='CA_ID')

# # 2.3 Elimina eventuale colonna chiave duplicata
# powbal_customer = powbal_customer.drop(columns=['CA_ID'])

# # 2.4 Coverage del merge
# matched = powbal_customer[['Lat','Lon']].notna().all(axis=1)
# print(f"Copertura coordinate dopo merge: {matched.mean():.2%}  "
#       f"({matched.sum():,}/{len(powbal_customer):,})")

# # 2.5 Mostra gli ID senza match (prime 10)
# unmatched_ids = (powbal_customer.loc[~matched, 'ca_number']
#                  .drop_duplicates().head(10))
# print("ID senza match (prime 10):")
# display(unmatched_ids.to_frame('ca_number'))

# # 2.6 Ricostruisci la WKT `location` del powbal (coerente con weather_data)
# powbal_customer['Lon'] = pd.to_numeric(powbal_customer['Lon'], errors='coerce')
# powbal_customer['Lat'] = pd.to_numeric(powbal_customer['Lat'], errors='coerce')
# powbal_customer['location'] = ("POINT (" 
#                                + powbal_customer['Lon'].round(6).astype(str) + " "
#                                + powbal_customer['Lat'].round(6).astype(str) + ")")


Copertura coordinate dopo merge: 58.83%  (4,348,020/7,391,239)
ID senza match (prime 10):


,ca_number
0,60000191183
148,60000254585
21803,60000318224
33835,60000319917
34477,60000333272
36388,60000334361
41109,60000335624
51588,60000340327
99670,60000501258
109511,60000550966


In [14]:
powbal_customer['city'].head()

KeyError: 'city'

In [23]:
# import pandas as pd
# import numpy as np

# # Copie di lavoro
# df_powbal = df_switch_transformed.copy()
# cust      = customer_data.copy()

# # --- POWBAL: sistema 'ca_number' ---
# # Se è float, rimuovo la parte decimale PRIMA di cast a stringa
# if pd.api.types.is_float_dtype(df_powbal['ca_number']):
#     # usa formattazione senza decimali; gestisce NaN
#     df_powbal['ca_number'] = df_powbal['ca_number'].apply(lambda x: ('%.0f' % x) if pd.notna(x) else x)

# # se è int, ok; in ogni caso converto a stringa e pulisco spazi "strani"
# df_powbal['ca_number'] = (
#     df_powbal['ca_number'].astype(str)
#     .str.replace('\u00A0|\u200B', '', regex=True)  # NBSP / zero-width
#     .str.strip()
# )

# # --- CUSTOMER: sistema 'CA_ID' ---
# cust['CA_ID'] = (
#     cust['CA_ID'].astype(str)
#     .str.replace('\u00A0|\u200B', '', regex=True)
#     .str.strip()
# )

# # Verifica univocità CA_ID (m:1 atteso)
# dups = cust['CA_ID'].duplicated(keep=False)
# if dups.any():
#     display(cust.loc[dups].sort_values('CA_ID').head(10))
#     raise ValueError("CA_ID NON è univoco nel customer: vedi sopra i duplicati.")


In [24]:
# # Colonne customer da portare
# cols_cust = ['CA_ID', 'Lat', 'Lon', 'city']

# # Se df_powbal ha già 'city', forziamo suffix per evitare city_x/city_y imprevedibili
# powbal_customer = df_powbal.merge(
#     cust[cols_cust],
#     how='left',
#     left_on='ca_number',
#     right_on='CA_ID',
#     validate='m:1',                # garantisce m:1
#     suffixes=('_pow', '_cust')     # suffix espliciti
# )

# # Elimina chiave duplicata
# powbal_customer = powbal_customer.drop(columns=['CA_ID'])

# # ---- UNIFICA 'city' ----
# # Preferiamo SEMPRE la city dal customer; se manca, fallback a quella (eventuale) in powbal
# has_city_pow  = 'city_pow'  in powbal_customer.columns
# has_city_cust = 'city_cust' in powbal_customer.columns

# if has_city_cust and has_city_pow:
#     powbal_customer['city'] = powbal_customer['city_cust'].where(
#         powbal_customer['city_cust'].notna(), powbal_customer['city_pow']
#     )
#     powbal_customer = powbal_customer.drop(columns=['city_cust','city_pow'])
# elif has_city_cust and not has_city_pow:
#     powbal_customer = powbal_customer.rename(columns={'city_cust':'city'})
# elif not has_city_cust and has_city_pow:
#     powbal_customer = powbal_customer.rename(columns={'city_pow':'city'})
# # se nessuno dei due: la colonna 'city' non esiste; in quel caso è un problema di customer → fermati e indaga
# if 'city' not in powbal_customer.columns:
#     raise KeyError("Dopo il merge non esiste 'city'. Controlla i nomi/suffix o la presenza della colonna nel customer.")

# # ---- Tipi e normalizzazione ----
# powbal_customer['city'] = powbal_customer['city'].astype(str).str.strip().str.lower()

# powbal_customer['Lat'] = pd.to_numeric(powbal_customer['Lat'], errors='coerce')
# powbal_customer['Lon'] = pd.to_numeric(powbal_customer['Lon'], errors='coerce')

# # Ricostruisci WKT location (coerente con weather)
# powbal_customer['location'] = (
#     "POINT (" + powbal_customer['Lon'].round(6).astype(str) + " " 
#               + powbal_customer['Lat'].round(6).astype(str) + ")"
# )

# # DEBUG: mostra quali colonne 'city' hai davvero e le prime righe
# print("Colonne che contengono 'city':", [c for c in powbal_customer.columns if 'city' in c.lower()])
# display(powbal_customer[['ca_number','city','Lat','Lon']].head(10))

# # Coverage coordinate (righe)
# matched = powbal_customer[['Lat','Lon']].notna().all(axis=1)
# print(f"Copertura coordinate dopo merge: {matched.mean():.2%}  ({matched.sum():,}/{len(powbal_customer):,})")

# # Eventuali ID senza match (campione)
# unmatched_ids = (powbal_customer.loc[~matched, 'ca_number'].drop_duplicates().head(10))
# print("ID senza match (prime 10):")
# display(unmatched_ids.to_frame('ca_number'))


Colonne che contengono 'city': ['city']


,ca_number,city,Lat,Lon
0,60000191183,delhi,NaN,NaN
1,60000191183,delhi,NaN,NaN
2,60000191183,delhi,NaN,NaN
3,60000191183,delhi,NaN,NaN
4,60000191183,delhi,NaN,NaN
5,60000191183,delhi,NaN,NaN
6,60000191183,delhi,NaN,NaN
7,60000191183,delhi,NaN,NaN
8,60000191183,delhi,NaN,NaN
9,60000191183,delhi,NaN,NaN


Copertura coordinate dopo merge: 58.83%  (4,348,020/7,391,239)
ID senza match (prime 10):


,ca_number
0,60000191183
148,60000254585
21803,60000318224
33835,60000319917
34477,60000333272
36388,60000334361
41109,60000335624
51588,60000340327
99670,60000501258
109511,60000550966
